# Notebook 03: Entrenamiento de Modelos Base y Extracción de Probabilidades con Entropía de Shannon

Este notebook cubre las siguientes etapas metodológicas de la tesis:
*   **Paso 9**: Entrenamiento de los modelos base de Nivel 0 (Random Forest, XGBoost y LightGBM) en el conjunto balanceado con Borderline-SMOTE.
*   **Paso 10**: Extracción de probabilidades de pertenencia a clase y cálculo de la **Entropía de Shannon** para medir la incertidumbre de las predicciones de los modelos base.

### Optimización de RAM para Entornos Locales (WSL2):
Para evitar caídas del servidor por falta de memoria (OOM), este notebook implementa:
1. **Muestreo Estratificado del 25%** del conjunto balanceado.
2. Lectura y procesamiento en precisión simple **`float32`**.
3. Restricción a **`n_jobs=1`** para evitar la creación de procesos hijos paralelos (Loky Forks) que duplican el uso de memoria RAM.

In [1]:
import os
import pandas as pd
import numpy as np
import json
import joblib
import time
import gc
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import xgboost as xgb
import lightgbm as lgb

# Ajustar directorio de trabajo si es necesario
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print(f"Directorio de trabajo activo: {os.getcwd()}")

sns.set_theme(style="whitegrid")
os.makedirs("data/processed", exist_ok=True)
os.makedirs("models", exist_ok=True)

Directorio de trabajo activo: /home/carmen/tesis_iot


In [2]:
# Cargar datos balanceados de entrenamiento y particiones de val/test escaladas en float32
print("Cargando particiones del conjunto de datos en float32...")
X_train_full = pd.read_csv("data/balanced/X_train_borderline.csv", dtype=np.float32)
y_train_full = pd.read_csv("data/balanced/y_train_borderline.csv").values.ravel().astype(np.int8)

X_val = pd.read_csv("data/processed/X_val.csv", dtype=np.float32)
y_val = pd.read_csv("data/processed/y_val.csv").values.ravel().astype(np.int8)

X_test = pd.read_csv("data/processed/X_test.csv", dtype=np.float32)
y_test = pd.read_csv("data/processed/y_test.csv").values.ravel().astype(np.int8)

with open("results/label_mapping.json", "r") as f:
    label_mapping = json.load(f)
class_names = [k for k, v in sorted(label_mapping.items(), key=lambda item: item[1])]

print(f"Full Train balanceado inicial: {X_train_full.shape[0]:,} muestras")

# Aplicar reducción por muestreo del 25% para evitar OOM
_, X_train, _, y_train = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=42, stratify=y_train_full
)

del X_train_full, y_train_full
gc.collect()

print(f"Entrenamiento ajustado en RAM: {X_train.shape[0]:,} muestras")
print(f"Validación: {X_val.shape[0]:,} muestras")
print(f"Prueba: {X_test.shape[0]:,} muestras")

Cargando particiones del conjunto de datos en float32...
Full Train balanceado inicial: 1,935,758 muestras
Entrenamiento ajustado en RAM: 483,940 muestras
Validación: 377,382 muestras
Prueba: 377,383 muestras


## Validación Cruzada (3 pliegues) y Generación de Probabilidades

In [3]:
oof_probs_rf = np.zeros((X_train.shape[0], 8))
oof_probs_xgb = np.zeros((X_train.shape[0], 8))
oof_probs_lgb = np.zeros((X_train.shape[0], 8))

val_probs_rf = np.zeros((X_val.shape[0], 8))
val_probs_xgb = np.zeros((X_val.shape[0], 8))
val_probs_lgb = np.zeros((X_val.shape[0], 8))

test_probs_rf = np.zeros((X_test.shape[0], 8))
test_probs_xgb = np.zeros((X_test.shape[0], 8))
test_probs_lgb = np.zeros((X_test.shape[0], 8))

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    start_fold = time.time()
    print(f"\n========== ENTRENANDO PLIEGUE {fold+1} / 3 ==========")
    
    X_tr = X_train.iloc[train_idx]
    y_tr = y_train[train_idx]
    X_va_fold = X_train.iloc[val_idx]
    
    # A. Random Forest (Secuencial para RAM)
    print("  -> Entrenando Random Forest (n_jobs=1)...")
    rf = RandomForestClassifier(n_estimators=30, max_depth=12, n_jobs=1, random_state=42)
    rf.fit(X_tr, y_tr)
    oof_probs_rf[val_idx] = rf.predict_proba(X_va_fold)
    val_probs_rf += rf.predict_proba(X_val) / 3
    test_probs_rf += rf.predict_proba(X_test) / 3
    
    # B. XGBoost GPU
    print("  -> Entrenando XGBoost...")
    xgb_model = xgb.XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.2,
        tree_method='hist', device='cuda', random_state=42
    )
    xgb_model.fit(X_tr, y_tr)
    oof_probs_xgb[val_idx] = xgb_model.predict_proba(X_va_fold)
    val_probs_xgb += xgb_model.predict_proba(X_val) / 3
    test_probs_xgb += xgb_model.predict_proba(X_test) / 3
    
    # C. LightGBM (Secuencial para RAM)
    print("  -> Entrenando LightGBM (n_jobs=1)...")
    lgb_model = lgb.LGBMClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        random_state=42, n_jobs=1, verbosity=-1
    )
    lgb_model.fit(X_tr, y_tr)
    oof_probs_lgb[val_idx] = lgb_model.predict_proba(X_va_fold)
    val_probs_lgb += lgb_model.predict_proba(X_val) / 3
    test_probs_lgb += lgb_model.predict_proba(X_test) / 3
    
    if fold == 0:
        joblib.dump(rf, "models/base_rf.pkl")
        joblib.dump(xgb_model, "models/base_xgb.pkl")
        joblib.dump(lgb_model, "models/base_lgb.pkl")
        print("  [Info] Modelos base guardados en 'models/'")
        
    # Limpieza manual
    del X_tr, y_tr, X_va_fold
    if fold > 0:
        del rf, xgb_model, lgb_model
    gc.collect()
    print(f"  Pliegue {fold+1} finalizado en {time.time() - start_fold:.2f} s.")


========== ENTRENANDO PLIEGUE 1 / 3 ==========
  -> Entrenando Random Forest (n_jobs=1)...
  -> Entrenando XGBoost...


/home/carmen/miniconda3/envs/tesis_iot/lib/python3.10/site-packages/xgboost/core.py:751: UserWarning: [01:01:53] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


  -> Entrenando LightGBM (n_jobs=1)...
  [Info] Modelos base guardados en 'models/'
  Pliegue 1 finalizado en 94.31 s.

========== ENTRENANDO PLIEGUE 2 / 3 ==========
  -> Entrenando Random Forest (n_jobs=1)...
  -> Entrenando XGBoost...
  -> Entrenando LightGBM (n_jobs=1)...
  Pliegue 2 finalizado en 96.94 s.

========== ENTRENANDO PLIEGUE 3 / 3 ==========
  -> Entrenando Random Forest (n_jobs=1)...
  -> Entrenando XGBoost...
  -> Entrenando LightGBM (n_jobs=1)...
  Pliegue 3 finalizado en 95.01 s.


## Cálculo de la Entropía de Shannon

In [4]:
def calcular_entropia_shannon(probabilidades):
    eps = 1e-15
    probabilidades = np.clip(probabilidades, eps, 1.0)
    return -np.sum(probabilidades * np.log2(probabilidades), axis=1)

entropy_tr_rf = calcular_entropia_shannon(oof_probs_rf)
entropy_tr_xgb = calcular_entropia_shannon(oof_probs_xgb)
entropy_tr_lgb = calcular_entropia_shannon(oof_probs_lgb)

entropy_val_rf = calcular_entropia_shannon(val_probs_rf)
entropy_val_xgb = calcular_entropia_shannon(val_probs_xgb)
entropy_val_lgb = calcular_entropia_shannon(val_probs_lgb)

entropy_te_rf = calcular_entropia_shannon(test_probs_rf)
entropy_te_xgb = calcular_entropia_shannon(test_probs_xgb)
entropy_te_lgb = calcular_entropia_shannon(test_probs_lgb)

## Construcción y Guardado de Meta-Features

In [5]:
def crear_meta_features(probs_rf, probs_xgb, probs_lgb, ent_rf, ent_xgb, ent_lgb):
    meta_df = pd.DataFrame()
    for c in range(8):
        meta_df[f'rf_prob_c{c}'] = probs_rf[:, c]
        meta_df[f'xgb_prob_c{c}'] = probs_xgb[:, c]
        meta_df[f'lgb_prob_c{c}'] = probs_lgb[:, c]
    meta_df['rf_entropy'] = ent_rf
    meta_df['xgb_entropy'] = ent_xgb
    meta_df['lgb_entropy'] = ent_lgb
    return meta_df

X_train_meta = crear_meta_features(oof_probs_rf, oof_probs_xgb, oof_probs_lgb, entropy_tr_rf, entropy_tr_xgb, entropy_tr_lgb)
X_val_meta = crear_meta_features(val_probs_rf, val_probs_xgb, val_probs_lgb, entropy_val_rf, entropy_val_xgb, entropy_val_lgb)
X_test_meta = crear_meta_features(test_probs_rf, test_probs_xgb, test_probs_lgb, entropy_te_rf, entropy_te_xgb, entropy_te_lgb)

X_train_meta.to_csv("data/processed/X_train_meta.csv", index=False)
pd.DataFrame(y_train, columns=['Label']).to_csv("data/processed/y_train_meta.csv", index=False)

X_val_meta.to_csv("data/processed/X_val_meta.csv", index=False)
pd.DataFrame(y_val, columns=['Label']).to_csv("data/processed/y_val_meta.csv", index=False)

X_test_meta.to_csv("data/processed/X_test_meta.csv", index=False)
pd.DataFrame(y_test, columns=['Label']).to_csv("data/processed/y_test_meta.csv", index=False)

print("Meta-features guardadas.")

Meta-features guardadas.


## Evaluación de Modelos Base Individuales (Sobre Validación)

In [6]:
for name, probs in [('Random Forest', val_probs_rf), ('XGBoost', val_probs_xgb), ('LightGBM', val_probs_lgb)]:
    preds = np.argmax(probs, axis=1)
    acc = accuracy_score(y_val, preds)
    print(f"\n--- Reporte de Validación: {name} (Accuracy: {acc*100:.2f}%) ---")
    print(classification_report(y_val, preds, target_names=class_names))


--- Reporte de Validación: Random Forest (Accuracy: 83.67%) ---
              precision    recall  f1-score   support

      Benign       0.65      0.53      0.58     18792
 Brute Force       0.40      0.39      0.39      1878
        DDoS       0.89      0.91      0.90    148475
         DoS       0.73      0.67      0.70     51945
       Mirai       1.00      0.99      1.00     55435
       Recon       0.74      0.87      0.80     62247
    Spoofing       0.90      0.70      0.79     35054
   Web-based       0.24      0.26      0.25      3556

    accuracy                           0.84    377382
   macro avg       0.69      0.67      0.68    377382
weighted avg       0.84      0.84      0.83    377382


--- Reporte de Validación: XGBoost (Accuracy: 84.84%) ---
              precision    recall  f1-score   support

      Benign       0.65      0.60      0.63     18792
 Brute Force       0.48      0.37      0.42      1878
        DDoS       0.89      0.92      0.90    148475
        